In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


# Training Walkthrough

This notebook runs the same staged training functions used by the CLI, step by step.

1. Configure the experiment.
2. Run ViT pretraining on BraTS tensors.
3. Run SSL pretraining initialized from the ViT pretraining checkpoint.
4. Run multimodal survival training and testing.

In [ ]:
from maikol_utils.print_utils import print_separator
from src.config import Configuration
from src.training import train_stage_vit_pretraining, train_stage_ssl, train_stage_survival, test_model
from src.utils import set_all_seeds
from scripts import prepare_data

CONFIG = Configuration(
    ssl_epochs=20,
    survival_epochs=20,
    masked_train=True,
    masked_test=True,
)

set_all_seeds(CONFIG.seed)


# Prepare data

In [3]:
print_separator("Preparing Data", sep_type='LONG')
ssl_train_loader, survival_dm, train_loader, val_loader, test_loader = prepare_data(CONFIG)

________________________________________________________________________________________________________________________________
                                                         Preparing Data                                                         

 - SSL Train samples:       1251
 - Survival Train samples:   482
 - Survival Val samples:      60
 - Survival Test samples:     61


## Step 1: ViT Pretraining

This runs a first pretraining stage on BraTS tensors before SSL.

In [4]:
print_separator("Starting ViT Pretraining", sep_type='SUPER')
# vit_pretraining_checkpoint = train_stage_vit_pretraining(CONFIG, ssl_train_loader)

                                                    Starting ViT Pretraining                                                    



## Step 2: SSL Pretraining

This stage initializes from the ViT pretraining checkpoint and continues SSL training.

In [5]:
print_separator("Starting SSL Pretraining", sep_type='SUPER')
ssl_checkpoint_path = train_stage_ssl(
    CONFIG,
    ssl_train_loader,
    # init_checkpoint_path=vit_pretraining_checkpoint,
)

                                                    Starting SSL Pretraining                                                    



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set `torch.set_fl

 - [SSL] device: gpu:0 (NVIDIA GeForce RTX 5070)


/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type           | Params | Mode  | FLOPs
-----------------------------------------------------------
0 | encoder | ViTEncoder3D   | 7.4 M  | train | 0    
1 | model   | SSLPretraining | 11.7 M | train | 0    
-----------------------------------------------------------
11.7 M    Trainable params
0         Non-trainable params
11.7 M    Total params
46.877    Total estimated model params size (MB)
57        Modules in train mode
0         Modules in eval mode
0         Total Flops
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.
FIT Profiler Report

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  Action                                                                                                                                                                	|  Mean duration (s)	|  Num calls      	|  Total time (s) 	|  Percentage %   	|
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  Total                                                                                                                           

 - Best SSL checkpoint: /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models/ssl_pretraining_best-v11.ckpt


## Step 3: Survival Training

This uses the Lightning `MultimodalSurvivalLightningModule` with the UPenn GBM data module.

In [6]:

print_separator("Starting Survival Pretraining", sep_type='SUPER')
survival_module = train_stage_survival(
    CONFIG, ssl_checkpoint_path,
    survival_dm, train_loader, 
    val_loader, test_loader
)

                                                 Starting Survival Pretraining                                                  



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type                        | Params | Mode  | FLOPs
--------------------------------------------------------------------------
0 | model     | MultimodalSurvivalPredictor | 10.0 M | train | 0    
1 | criterion | CrossEntropyLoss            | 0      | train | 0    
--------------------------------------------------------------------------
10.0 M    Trainable params
0         Non-trainable params
10.0 M    Total params
40.191    Total estimated model params size (MB)
97        Modules in train mode
0         Modules in eval mode
0         Total Flops


[pretrained encoder] missing=0  unexpected=0
 - [Survival] device: gpu:0 (NVIDIA GeForce RTX 5070)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.6721311211585999
        test_bacc           0.6712301969528198
        test_loss           0.6445956230163574
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 - Best Survival checkpoint: /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models/survival_checkpoint_best-v8.ckpt


In [7]:
print_separator("Step 4: Testing", sep_type='SUPER')
results = test_model(CONFIG, survival_module, test_loader)

                                                        Step 4: Testing                                                         



Testing: 100%|██████████| 4/4 [00:02<00:00,  1.85it/s]

 - Test F1 Score:  0.6671
 - Test Precision: 0.6862
 - Test Recall:    0.6721
 - Test Accuracy:  0.6721


In [8]:
results['predictions']

[np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0)]